In [ ]:
# -------------------------------------
import os, math, csv, random, numpy as np
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" # Disable GPU usage if not needed or causing issues

import tensorflow as tf
import matplotlib.pyplot as plt
import sionna
from sionna.rt import (load_scene, PlanarArray, Transmitter, Receiver,
                       PathSolver, subcarrier_frequencies, scene) # Added scene to imports for sionna.rt.scene.etoile

# ---------- 1. 場景 & 天線 (Scene & Antennas) ---------------------------------------------
# Load a predefined scene from Sionna
# The paper uses a custom setup, 'etoile' is a placeholder.
# For full reproduction, the scene geometry should match the paper's description.
current_scene = load_scene(sionna.rt.scene.etoile)

# Set carrier frequency according to the paper (3.5 GHz)
current_scene.frequency = 5e9 # Changed from 30e9

# Configure transmitter (BS) antenna array
# Paper: BS equipped with an 8-column antenna array, each column consists of 4 channels.
# Horizontal spacing dh = 0.5 lambda, vertical spacing dv = 2.0 lambda. Cross-polarization.
# This results in 4 rows, 8 columns, dual-polarized (2 ports per element) = 4*8*2 = 64 effective antennas/ports.
current_scene.tx_array = PlanarArray(num_rows=4,
                                     num_cols=8,
                                     horizontal_spacing=0.5, # In wavelengths
                                     vertical_spacing=2.0,   # In wavelengths
                                     pattern="tr38901",      # 3GPP antenna pattern
                                     polarization="cross")   # Enables dual polarization (V and H)

# Configure receiver (UE) antenna array
# Paper: UE antennas are 2 (implied from CSI dimensions 2x62x408)
# A 1x1 array with "cross" polarization provides 2 effective antenna ports.
current_scene.rx_array = PlanarArray(num_rows=1,
                                     num_cols=1,
                                     horizontal_spacing=0.5,
                                     vertical_spacing=0.5,
                                     pattern="tr38901",
                                     polarization="cross")

# Remove default BS and UE if they exist from the loaded scene
if "bs" in current_scene.objects:
    current_scene.remove("bs")
if "ue" in current_scene.objects:
    current_scene.remove("ue")

# Add Transmitter (BS) at the position specified in the paper (Table I: (100, -100, 30))
# Note: Sionna's coordinate system might need verification against the paper's assumptions.
current_scene.add(Transmitter("bs", [100., -100., 30.])) # Changed from [10, 50, 80]

# Add Receiver (UE) at a sample position.
# Paper: UEs distributed across three sectors at a height of 1.5 m.
# For a single UE plot, we use a placeholder position.
current_scene.add(Receiver("ue", [-50., -40., 15])) # Changed Z-coordinate from 15 to 1.5

: 

In [ ]:


# ---------- 2. 頻率軸 & 路徑求解器 (Frequency Axis & Path Solver) --------------------------------------
# Configure ray tracing parameters
# Paper: Fibonacci method, 10,000 rays, max interaction count 4 (max_depth=5 if LoS is depth 1)
# Only direct and reflected paths considered.


solver = PathSolver() # Pass scene to solver constructor
paths = solver(scene=current_scene,max_depth=0, # Consistent with "interaction count 4"
               los=True,
               specular_reflection=True,
               diffuse_reflection=False, # Paper does not explicitly mention diffuse
               refraction=False,         # Paper does not explicitly mention refraction
               synthetic_array=True)     # Treats array as a single point for faster ray tracing

# OFDM parameters from the paper / typical values
# Paper: Subcarrier spacing 30 kHz. SRS adopts 2-comb structure -> 60 kHz between SRS REs.
# Bandwidth 100 MHz -> 1632 SRS REs.
# Down-sampling 4-to-1 -> 408 SRS REs, effective subcarrier spacing 240 kHz for these REs.
NUM_SC_TOTAL = 1632 # Total SRS REs over 100 MHz with 60kHz effective spacing
SAMPLE_FACTOR = 4   # Down-sampling factor
NUM_SC = NUM_SC_TOTAL // SAMPLE_FACTOR # Final number of subcarriers for CSI (408)
# Effective spacing between the 1632 SRS REs is 60kHz
SRS_RE_SPACING = 60e3
# The subcarrier spacing for the *final* 408 samples, after downsampling.
# This is also NUM_SC_TOTAL * SRS_RE_SPACING / NUM_SC, or SRS_RE_SPACING * SAMPLE_FACTOR
EFFECTIVE_SC_SPACING_AFTER_DS = SRS_RE_SPACING * SAMPLE_FACTOR # 240 kHz

# NUM_SYM is the number of OFDM symbols for time evolution if needed, or for Doppler calculation.
# The paper's target CSI (2x62x408) seems to be for a single time snapshot.
NUM_SYM = 1 # For a single snapshot CSI, as implied by final dimensions in paper.
            # If time evolution was needed, this would be > 1 (e.g., 1024 as in your original code)

# Frequencies for CFR calculation: NUM_SC_TOTAL points with SRS_RE_SPACING
freqs_for_cfr = subcarrier_frequencies(NUM_SC_TOTAL, SRS_RE_SPACING)

# Calculate Channel Frequency Response (CFR)
# sampling_frequency for time evolution: NUM_SYM * SRS_RE_SPACING (if NUM_SYM > 1)
# For NUM_SYM = 1, this parameter's impact on a single snapshot needs care.
# Let's use a nominal sampling frequency related to the OFDM structure.
# The paper mentions "num_time_steps=NUM_SYM" for paths.cfr.
# If NUM_SYM=1, it means we are taking the channel at one point in time.
h_full = paths.cfr(frequencies=freqs_for_cfr,
                   # A nominal sampling frequency for the time domain processing,
                   # if NUM_SYM > 1 this would be NUM_SYM * EFFECTIVE_SC_SPACING_AFTER_DS or similar.
                   # For NUM_SYM=1, its role is less direct for the snapshot.
                   # Let's use a value that would correspond to a typical OFDM symbol duration.
                   sampling_frequency=NUM_SYM * SRS_RE_SPACING, # Or NUM_SYM * some_ofdm_symbol_rate
                   num_time_steps=NUM_SYM,
                   normalize_delays=False,
                   normalize=False,
                   out_type="tf")

print(f"Shape of h_full (raw CFR): {h_full.shape}")
# Expected: [batch_size, num_rx_obj, num_rx_ant_ue, num_tx_obj, num_tx_ant_bs, num_time_steps, num_subcarriers_total]
# With 1 UE, 1 BS: [1, 1, 2 (UE_pol), 1, 64 (BS_pol), NUM_SYM, NUM_SC_TOTAL]
# -> [1, 1, 2, 1, 64, 1, 1632]
# Squeeze out batch_size, num_rx_obj, num_tx_obj dimensions
h_squeezed = h_full[0, :, 0, :, 0, :] # Squeeze out batch and object dimensions)
print(f"Shape of h_squeezed: {h_squeezed.shape}")
# Expected: [num_rx_ant_ue, num_tx_ant_bs, num_time_steps, num_subcarriers_total]
# -> [1,2, 64, 1, 1632]

# Down-sample in the subcarrier dimension
h_ds = h_squeezed[..., ::SAMPLE_FACTOR]
print(f"h_ds shape after subcarrier downsampling: {h_ds.shape}")
# Expected: [num_rx_ant_ue, num_tx_ant_bs, num_time_steps, NUM_SC (408)]
# -> [2, 64, 1, 408]

# Select the first (and only) time step (t=0) and 62 out of 64 BS antenna/polarization channels
# Paper's CSI dimension is (2 x 62 x 408)
h_t0 = h_ds[:, :62,  :] # Select 62 BS antennas from the 64 available
print(f"h_t0 shape (spatial-frequency domain CSI): {h_t0.shape}")
# Expected: [num_rx_ant_ue, 62 (selected_bs_ant), NUM_SC (408)]
# -> [2, 62, 408]

# Transform to Angle-Delay domain
# 1. IFFT along subcarrier dimension (frequency to delay)
h_delay = tf.signal.ifft(h_t0) # Axis -1 is default (subcarriers)
# Shape: [2, 62, 408_delays]

# 2. FFT along BS antenna dimension (spatial to angle)
# Transpose to bring BS antenna dimension to the last for FFT
h_angle_tmp = tf.transpose(h_delay, perm=[0, 2, 1]) # Shape: [2, 408_delays, 62_bs_ant]
h_angle_tmp = tf.signal.fft(h_angle_tmp) # FFT along the last dimension (BS antennas)
h_angle_tmp = tf.signal.fftshift(h_angle_tmp, axes=2) # Shift zero-frequency component to center for angle
# Transpose back
h_ang_del = tf.transpose(h_angle_tmp, perm=[0, 2, 1]) # Shape: [2, 62_angles, 408_delays]
print(f"h_ang_del shape (angle-delay domain CSI): {h_ang_del.shape}")

# Prepare CSI for neural network input (real and imaginary parts)
real_part = tf.math.real(h_ang_del)
imag_part = tf.math.imag(h_ang_del)
# Paper mentions "converting the complex CSI into its magnitude and phase components"
# or concatenating real/imag. Concatenation is common.
# csi_nn_input = tf.stack([real_part, imag_part], axis=0) # Example: (2, 2, 62, 408)
csi_nn_input = tf.concat([real_part, imag_part], axis=0) # Shape: (4, 62, 408) if concat along new axis 0
                                                       # Or (2, 2*62, 408) or (2, 62, 2*408) depending on model
print(f"Real part shape: {real_part.shape}, Imaginary part shape: {imag_part.shape}")
print(f"Final CSI for NN input (concatenated real/imag): {csi_nn_input.shape}")


# --- Plotting one UE's angle-delay CSI 3D surface ---
# Convert to NumPy for plotting
real_np = real_part.numpy()
imag_np = imag_part.numpy()

# Select one UE polarization for plotting (e.g., the first one, index 0)
# Your original code used index 1: r0 = real[1]; i0 = imag[1]
ue_pol_idx_to_plot = 0
r0 = real_np[ue_pol_idx_to_plot] # Shape: (62, 408)
i0 = imag_np[ue_pol_idx_to_plot] # Shape: (62, 408)

# Calculate power |H|^2
power = np.abs(r0 + 1j*i0)**2 # Shape: (62, 408)

N_angle_bins, N_delay_bins = power.shape
print(f"Plotting power surface with {N_angle_bins} angle bins and {N_delay_bins} delay bins.")

# Create meshgrid for plotting
X_delay, Y_angle = np.meshgrid(np.arange(N_delay_bins), np.arange(N_angle_bins))

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(X_delay, Y_angle, power, cmap='viridis')

ax.set_xlabel("Delay Bin Index") # Updated label
ax.set_ylabel("Angle Bin Index") # Updated label
ax.set_zlabel("|H(angle, delay)|² (linear)")
ax.set_title(f"Angle-Delay Power Profile (UE Polarization {ue_pol_idx_to_plot})")

fig.colorbar(surf, shrink=0.5, aspect=10, pad=0.1, label='Power |H|²')
plt.show()

# --- To generate a dataset ---
# You would typically loop the following:
# 1. Set new UE positions: current_scene.receivers["ue"].position = [new_x, new_y, 1.5]
# 2. Re-run solver: paths = solver(...)
# 3. Re-calculate CFR and process to h_ang_del
# 4. Store h_ang_del (or its real/imag parts) and the UE coordinates.
# 5. Implement TA and AWGN as per the paper's description of practical imperfections.
#    - TA: Apply a phase shift in the frequency domain (h_t0) or a circular shift in the delay domain (h_delay).
#    - AWGN: Add complex Gaussian noise to h_t0 or h_ang_del.
current_scene.preview(paths=paths)  # Visualize the scene with paths


Dr.Jit encountered an unrecoverable error and will now shut
down. Please re-run your program in debug mode to check for
out-of-bounds reads, writes, and other sources of undefined
behavior. You can do so by calling

   dr.set_flag(dr.JitFlag.Debug, True)

at the beginning of the program. If these additional checks
fail to pinpoint the problem, then you have likely found a
bug. We are happy to help investigate and fix the problem if
you can you create a self-contained reproducer and submit it
at https://github.com/mitsuba-renderer/drjit.

The error message of this specific failure is as follows:
>>> jit_llvm_compile(): module could not be verified! Please see the LLVM IR and error message below:

define void @drjit_568b3cd8f91c4e0d3cc32536d1888b6a(i64 %start, i64 %end, i32 %thread_id, ptr noalias %params) #0 {
entry:
    %callables = load ptr, ptr @callables, align 8
    %buffer = alloca i8, i32 1280, align 32
    br label %body

body:
    %index = phi i64 [ %index_next, %suffix ], [ %